# 01 - Data ExplorationExploratory analysis of the **synthetic development dataset** used by theSmart Energy Monitoring & Load Prediction System.> The data is generated by `src/data/generate_data.py` from a mathematical load> model. It is **not** measured with a real energy meter. It exists so the> pipeline can be developed and tested without hardware.**Questions this notebook answers**1. What does the raw data look like, and is it internally consistent?2. Does the load follow a realistic daily and weekly rhythm?3. Are the electrical relationships (P = V·I·PF) satisfied?4. How much energy is consumed, and when are the peaks?

In [ ]:
import sysfrom pathlib import Path# Make the project importable when the notebook runs from notebooks/ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snssns.set_theme(style="whitegrid")plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
from src.data.generate_data import generate_energy_data, summarisefrom src.data.preprocess import validate_data, clean_datadf = generate_energy_data()          # reproducible: fixed seedprint(summarise(df))df.head()

## 1. ValidationNothing is cleaned silently. The validator reports what it found; the cleanerreports what it changed.

In [ ]:
report = validate_data(df)print(report.as_text())

In [ ]:
df.describe().T

## 2. Load over timeTwo views: the whole record (to see seasonality) and a single week (to see thedaily rhythm).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7))axes[0].plot(df["timestamp"], df["active_power"], lw=0.4)axes[0].set_title("Active power - full record")axes[0].set_ylabel("P (W)")week = df[(df["timestamp"] >= "2025-03-03") & (df["timestamp"] < "2025-03-10")]axes[1].plot(week["timestamp"], week["active_power"], lw=1.2, color="tab:orange")axes[1].set_title("Active power - one week")axes[1].set_ylabel("P (W)")plt.tight_layout()plt.show()

## 3. Daily and weekly load profileThe evening peak is the defining feature of a domestic load curve: lighting,cooking and entertainment coincide after working hours.

In [ ]:
tmp = df.copy()tmp["hour"] = tmp["timestamp"].dt.hourtmp["dow"] = tmp["timestamp"].dt.day_name().str[:3]fig, axes = plt.subplots(1, 2, figsize=(13, 4))tmp.groupby("hour")["active_power"].mean().plot(kind="bar", ax=axes[0], color="tab:blue")axes[0].set_title("Average load by hour of day")axes[0].set_ylabel("P (W)")order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]tmp.groupby("dow")["active_power"].mean().reindex(order).plot(    kind="bar", ax=axes[1], color="tab:green")axes[1].set_title("Average load by day of week")plt.tight_layout()plt.show()

## 4. Electrical consistencyEvery generated row must satisfy **P = V × I × PF**. Any drift here would meanthe generator (or the preprocessing) has a bug.

In [ ]:
expected = df["voltage"] * df["current"] * df["power_factor"]residual = df["active_power"] - expectedprint(f"max |residual| = {residual.abs().max():.4f} W")fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))axes[0].hist(df["voltage"], bins=50, color="tab:blue")axes[0].set_title("Voltage (V)")axes[1].hist(df["power_factor"], bins=50, color="tab:orange")axes[1].set_title("Power factor")axes[2].scatter(df["active_power"], df["voltage"], s=2, alpha=0.2)axes[2].set_title("Voltage sag vs load")axes[2].set_xlabel("P (W)"); axes[2].set_ylabel("V")plt.tight_layout(); plt.show()

Note the negative slope in the third panel: heavier current draw pulls thesupply voltage down, which is exactly what a real feeder does.## 5. Energy and peaks

In [ ]:
from src.utils import analyticsdaily = analytics.daily_consumption(df)summary = analytics.consumption_summary(df)peaks = analytics.peak_analysis(df)print(f"Average daily consumption : {summary['avg_daily_kwh']:.2f} kWh")print(f"Maximum daily consumption : {summary['max_daily_kwh']:.2f} kWh")print(f"Peak load                 : {peaks['max_load_w']:.0f} W at {peaks['peak_timestamp']}")print(f"Peak hour of day          : {peaks['peak_hour_of_day']}:00")print(f"Load factor               : {peaks['load_factor']:.3f}")daily.plot(x="timestamp", y="energy_kwh", legend=False, title="Daily energy (kWh)")plt.ylabel("kWh"); plt.show()

## 6. Correlation structure`energy` is a linear transform of `active_power`, so its correlation is 1.0 byconstruction — which is exactly why it must be excluded from the model features(see notebook 02).

In [ ]:
corr = df[["voltage", "current", "power_factor", "frequency", "active_power", "energy"]].corr()sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)plt.title("Correlation matrix"); plt.show()

## Takeaways- The dataset reproduces a realistic domestic load curve with an evening peak.- P = V·I·PF holds to numerical precision.- Voltage sags under load, and power factor improves with load.- `energy` and `active_power` are perfectly collinear → leakage risk for modelling.